In [2]:
pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pyodbc
import pandas as pd

# Connect to SQL Server
conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=DepressionAnalysisDB;"
    "Trusted_Connection=yes;"
)

# Query data
query = "SELECT * FROM depression_analysis_processed"

df = pd.read_sql(query, conn)

print(df.head())

   age  gender  stress  anxiety  sadness  overwhelmed  concentration  \
0   21  Female       4        3        3            2              2   
1   21    Male       2        2        3            0              2   
2   22  Female       0        2        2            2              0   
3   21    Male       4        0        4            3              2   
4   22    Male       3        3        3            0              2   

   overthinking  mood_swings  tired  ...  sleep  emotional_score  \
0             3            3      2  ...      1         3.333333   
1             4            2      2  ...      1         2.333333   
2             1            3      2  ...      2         1.333333   
3             3            3      0  ...      2         2.666667   
4             2            0      2  ...      1         3.000000   

   cognitive_score  sleep_risk  physical_score  social_score  \
0         2.333333           1             1.5           3.0   
1         2.000000           1

C:\Users\MS994\AppData\Local\Temp\ipykernel_24036\95901306.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [ ]:

# 3. DATA PREPROCESSING
# ==============================

# Drop ID column


# 🚨 Remove derived score columns (avoid data leakage)
drop_cols = [
    "emotional_score",
    "cognitive_score",
    "sleep_risk",
    "physical_score",
    "social_score",
    "behavioral_score",
    "drs_raw",
    "drs"
]

df = df.drop(columns=drop_cols)

# Separate target and features
y = df["risk_level"]
X = df.drop(columns=["risk_level"])

# Encode categorical column (gender)
X = pd.get_dummies(X, columns=["gender"], drop_first=True)

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)


# ==============================
# 4. TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ==============================
# 5. TRAIN MODEL
# ==============================
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)


# ==============================
# 6. EVALUATE MODEL
# ==============================
y_pred = rf.predict(X_test)

print("\nModel Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


# ==============================
# 7. FEATURE IMPORTANCE (OPTIONAL BUT IMPORTANT)
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

importances = rf.feature_importances_
features = X.columns

feat_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x="Importance", y="Feature", data=feat_df.head(10))
plt.title("Top 10 Important Features")
plt.show()


Model Evaluation:
Accuracy: 0.8757763975155279

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        39
           1       0.93      0.60      0.73        45
           2       0.81      0.97      0.88        77

    accuracy                           0.88       161
   macro avg       0.91      0.86      0.87       161
weighted avg       0.89      0.88      0.87       161

